# 5장. AI 예측 모델로 후크의 응력/변위 학습하기

## 개요

**목표:** PhysicsNeMo Transolver로 후크 학습 → 검증 총손실로 모델 선택 → 에폭 수/학습 노드 수의 영향 비교

| 순서 | 배우고 실행할 것 |
|---|---|
| 1. 학습 문제와 자료 | 입력/정답, 분할, 정규화 기준 확인 |
| 2. 후크용 Transolver | 좌표/조건 전달과 출력 채널 연결 |
| 3. 한 배치의 학습 | 예측/손실/역전파/가중치 조정 |
| 4. 먼저 100에폭 학습 | GPU 학습, 고정 검증 노드, 기록 |
| 5. 100에폭 결과 확인 | 곡선/응력/변위 오차와 부족한 부분 관찰 |
| 6. 같은 조건으로 1,000에폭 학습 | 학습 횟수만 늘려 다시 실행 |
| 7. 두 실행 비교 | 정답/두 예측, 오차/최대응력/시간 비교 |
| 8. 학습 노드 수 비교 | 케이스 1개씩, 4,096개 노드와 전체 노드 학습 비교 |
| 9. 결과 정리 | 검증으로 선택한 모델과 6장 연결 |


## 1. 학습 문제와 자료 확인


### 1.2 노드 샘플링과 배치


### 05장 / 셀 03 / Notebook


In [ ]:
# 1. 교육 폴더를 찾고 데이터/모델/학습 도구를 가져옵니다.
from pathlib import Path
from datetime import datetime
from dataclasses import replace
import copy, json, sys
import torch
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Image
working_dir = Path.cwd().resolve()
education_dir = next((p for base in [working_dir, *working_dir.parents]
                      for p in [base, base / 'education']
                      if (p / 'course_utils/dataset.py').is_file()), None)
if education_dir is None:
    raise FileNotFoundError('교육 폴더 또는 그 하위 폴더에서 실행하세요.')
if str(education_dir) not in sys.path:
    sys.path.insert(0, str(education_dir))
from course_utils.dataset import load_prepared
from course_utils.hook_transolver import make_model, MODEL_CONFIG
from course_utils.training import TrainingConfig, seed_training, training_loaders
from course_utils.training_transolver import run_epoch, train_model
from course_utils.checkpoint import load_checkpoint
from course_utils.transolver_plot import plot_history, plot_validation_case
from course_utils.transolver_compare import check_comparison, compare_case, plot_comparison
print('교육 폴더:', education_dir)


### 05장 / 셀 05 / Notebook


In [ ]:
# 3장의 분할/정규화 기준으로 자료를 읽고 배치 하나를 확인합니다.
cases, split, normalizer, fixed, dataset_manifest = load_prepared(
    education_dir / 'data', education_dir / 'runs/dataset')
config = replace(TrainingConfig(), max_epochs=100, log_every=10)
seed_training(config.seed)
datasets, loaders = training_loaders(cases, split, normalizer, fixed, config)
batch = next(iter(loaders['train']))
print('학습:', split['train'], '| 검증:', split['validation'])
print('DataLoader:', list(loaders), '| 시험 DataLoader 없음')
for k in ['coords','globals','stress','displacement']:
    print(k, tuple(batch[k].shape))
assert set(loaders) == {'train','validation'}


### 05장 / 셀 07 / Notebook


In [ ]:
# 한글 글꼴과 그래프 눈금을 설정합니다.
font_path = Path('C:/Windows/Fonts/malgun.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'dejavusans'
print('그림 표시 설정 완료')


## 2. 후크용 Transolver 구성


### 2.2 GPU에서 실제 배치 확인


### 05장 / 셀 10 / Notebook


In [ ]:
# 1. GPU에서 새 후크용 Transolver를 만듭니다.
if not torch.cuda.is_available():
    raise RuntimeError('1장의 GPU/CUDA용 PyTorch 점검을 완료하세요.')
device = torch.device('cuda')
seed_training(config.seed)
initial_model = make_model(MODEL_CONFIG).to(device).eval()
tensors = {k: batch[k].to(device) for k in ['coords','globals','stress','displacement']}
# 2. 같은 두께/하중을 각 노드에 연결하고 실제 제공 모델로 계산합니다.
point_conditions = tensors['globals'][:,None,:].expand(-1, tensors['coords'].shape[1], -1)
with torch.no_grad():
    native_output = initial_model.transolver(point_conditions, embedding=tensors['coords'])
    named_output = initial_model((tensors['coords'], tensors['globals']))
print('모델:', type(initial_model.transolver).__module__+'.'+type(initial_model.transolver).__name__)
print('GPU:', torch.cuda.get_device_name(device))
print('점별 두께/하중:', tuple(point_conditions.shape), '| 원본 출력:', tuple(native_output.shape))
print('응력:', tuple(named_output['stress'].shape), '| 변위:', tuple(named_output['displacement'].shape))
assert native_output.shape == (2,4096,4)
torch.testing.assert_close(named_output['stress'], native_output[...,:1])
torch.testing.assert_close(named_output['displacement'], native_output[...,1:])


## 3. 한 배치의 학습 과정 이해


### 3.2 응력/변위 손실과 한 배치 확인


### 05장 / 셀 12 / Notebook


In [ ]:
# 1. 고정 검증 자료에서 학습 전 손실을 읽습니다.
initial_validation = run_epoch(initial_model, loaders['validation'], device, normalizer)
print('학습 전 검증 총손실:', initial_validation['loss']['total'])

# 2. 초기 모델 복사본과 가중치 조정 도구를 준비합니다.
from course_utils.models import normalized_loss
probe_model = copy.deepcopy(initial_model)
before = {k: v.detach().clone() for k, v in probe_model.state_dict().items()}
probe_optimizer = torch.optim.AdamW(probe_model.parameters(),
    lr=config.learning_rate, weight_decay=config.weight_decay)
tensors = {k: batch[k].to(device) for k in ['coords', 'globals', 'stress', 'displacement']}

# 3. 한 배치로 예측 → 손실 → 역전파 → 가중치 조정을 수행합니다.
probe_model.train()
probe_optimizer.zero_grad(set_to_none=True)
prediction = probe_model((tensors['coords'], tensors['globals']))
losses = normalized_loss(prediction, tensors)
losses['total'].backward()
torch.nn.utils.clip_grad_norm_(probe_model.parameters(), config.gradient_clip, error_if_nonfinite=True)
probe_optimizer.step()

# 4. 손실은 숫자로 꺼내고, 실제로 가중치가 바뀌었는지 확인합니다.
changed = sum(not torch.equal(before[k], v) for k, v in probe_model.state_dict().items())
print('한 배치의 조정 전 손실:', {k: v.detach().item() for k, v in losses.items()})
print('변경된 가중치 배열 수:', changed)
assert changed > 0
del probe_model, probe_optimizer, prediction, losses, before


## 4. 먼저 100에폭 학습하기


### 4.3 100에폭 실행과 저장


### 05장 / 셀 14 / Notebook


In [ ]:
# 1. 새 모델로 학습하고 매 에폭 검증합니다.
run_dir = education_dir / 'runs/training' / ('student_transolver_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
summary = train_model(education_dir, run_dir, config=config, device='cuda', controls=False)
# 2. 이번 실행의 결과를 읽습니다.
print('실행 폴더:', run_dir, '| 선택 에폭:', summary['best_epoch'])
print('초기 / 선택 검증 손실:', summary['initial_validation_loss']['total'], summary['best_validation_loss']['total'])
print('검증으로 선택한 모델의 검증 MAE:', summary['best_validation_mae'])
print('실행 시간(초):', round(summary['elapsed_seconds'],2))
assert summary['status'] == 'completed' and not summary['test_scored']

# 3. 뒤의 비교에서 사용할 100에폭 실행 정보를 따로 보관합니다.
run_dir_100 = run_dir
summary_100 = summary


## 5. 100에폭 결과에서 부족한 부분 찾기


### 5.1 학습 곡선과 물리 단위 오차


### 05장 / 셀 16 / Notebook


In [ ]:
# 본인 실행의 손실과 물리 단위 MAE를 표시합니다.
curve_path = plot_history(run_dir, run_dir / 'learning_curves.png')
display(Image(filename=str(curve_path)))


### 5.3 100에폭 실행에서 선택한 모델 다시 불러오기


### 05장 / 셀 19 / Notebook


In [ ]:
# 1. best 파일을 읽고 같은 고정 검증 노드에서 확인합니다.
best_model, saved_normalizer, package = load_checkpoint(run_dir / 'best_model.pt', device,
    expected_split_sha256=split['split_sha256'])
before = {k:v.detach().clone() for k,v in best_model.state_dict().items()}
reloaded = run_epoch(best_model, loaders['validation'], device, saved_normalizer)
reload_difference = max(abs(reloaded['loss'][k]-summary['best_validation_loss'][k]) for k in reloaded['loss'])
print('모델 종류:', package['model_kind'], '| 선택 에폭:', package['training']['epoch'])
print('검증 재로딩 차이:', reload_difference, '| 검증 MAE:', reloaded['mae'])
assert reload_difference < 1e-6 and saved_normalizer.stats == normalizer.stats
assert all(torch.equal(before[k],v) for k,v in best_model.state_dict().items())
assert all(p.grad is None for p in best_model.parameters())


### 5.4 100에폭 실행의 응력 분포 확인


### 05장 / 셀 21 / Notebook


In [ ]:
# 첫 검증 케이스의 전체 메시를 표시합니다. 시험 자료를 사용하지 않습니다.
validation_case = cases[split['validation'][0]]
mesh_result = plot_validation_case(education_dir,best_model,saved_normalizer,validation_case,
    run_dir / 'validation_stress.png', field='stress')
print('그림 대상:',mesh_result['case_id'],'| 전체 노드:',mesh_result['nodes'])
print('전체 노드 응력 MAE(MPa):',mesh_result['full_node_mae'])
print('ANSYS / 예측 최대응력(MPa):',mesh_result['maximum_target'],mesh_result['maximum_prediction'])
display(Image(filename=mesh_result['image']))


## 6. 같은 조건으로 1,000에폭 학습하기


### 6.2 새 실행 폴더에서 학습


### 05장 / 셀 24 / Notebook


In [ ]:
# 1. 학습 횟수만 바꾸고 같은 초기 조건에서 다시 시작합니다.
# log_every는 화면 출력 간격이며 가중치 조정/검증 횟수에는 영향을 주지 않습니다.
long_config = replace(config, max_epochs=1000, log_every=100)
run_dir_1000 = education_dir / 'runs/training' / ('student_transolver_1000_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
summary_1000 = train_model(education_dir, run_dir_1000, config=long_config, device='cuda', controls=False)
# 2. 실제로 같은 조건에서 비교했는지 검사합니다.
comparison_check = check_comparison(run_dir_100, run_dir_1000)
print('100에폭과 1,000에폭 실행 조건 확인:', comparison_check)
print('1,000에폭 실행의 선택 에폭:', summary_1000['best_epoch'])
print('1,000에폭 실행의 검증 MAE:', summary_1000['best_validation_mae'])
print('1,000에폭 학습 시간(초):', round(summary_1000['elapsed_seconds'],2))
assert not summary_1000['test_scored']


### 6.3 1,000에폭 실행의 학습 곡선


### 05장 / 셀 26 / Notebook


In [ ]:
# 1,000에폭 실행에서 기록한 손실/MAE를 표시합니다.
long_curve_path = plot_history(run_dir_1000, run_dir_1000 / 'learning_curves.png')
display(Image(filename=str(long_curve_path)))


## 7. 100에폭과 1,000에폭 결과 비교


### 7.1 같은 기준의 숫자로 비교하기


### 05장 / 셀 28 / Notebook


In [ ]:
# 1. 1,000에폭 실행에서 선택한 파일을 읽고 고정 검증 노드에서 재확인합니다.
long_model, long_normalizer, long_package = load_checkpoint(run_dir_1000 / 'best_model.pt', device,
    expected_split_sha256=split['split_sha256'])
long_reloaded = run_epoch(long_model, loaders['validation'], device, long_normalizer)
long_reload_difference = max(abs(long_reloaded['loss'][k]-summary_1000['best_validation_loss'][k]) for k in long_reloaded['loss'])
assert long_reload_difference < 1e-6 and long_normalizer.stats == saved_normalizer.stats
print('1,000에폭 실행에서 검증으로 선택한 모델의 재로딩 차이:', long_reload_difference)

# 2. 두 실행의 고정 검증 노드 지표와 전체 학습 시간을 비교합니다.
print('지표 | 100에폭 실행 | 1,000에폭 실행')
for label, a, b in [
    ('선택 에폭',summary_100['best_epoch'],summary_1000['best_epoch']),
    ('검증 총손실',summary_100['best_validation_loss']['total'],summary_1000['best_validation_loss']['total']),
    ('검증 응력 MAE(MPa)',summary_100['best_validation_mae']['stress_MPa'],summary_1000['best_validation_mae']['stress_MPa']),
    ('전체 학습 시간(초)',summary_100['elapsed_seconds'],summary_1000['elapsed_seconds'])]:
    print(f'{label} | {a:.6g} | {b:.6g}')
for axis in 'XYZ':
    key=f'displacement_{axis}_mm'
    print(f"검증 변위 {axis} MAE(mm) | {summary_100['best_validation_mae'][key]:.6g} | {summary_1000['best_validation_mae'][key]:.6g}")

# 3. 같은 검증 케이스의 전체 노드를 각각 한 번에 전달해 비교합니다.
comparison_outputs, comparison_rows = compare_case(best_model,long_model,saved_normalizer,validation_case)
print('전체 메시 비교 대상:',validation_case.case_id,'| 노드:',len(validation_case.coords))
comparison_record={'conditions':comparison_check,'case_metrics':comparison_rows,
    'run_100':str(run_dir_100),'run_1000':str(run_dir_1000),'test_scored':False}
(run_dir_1000 / 'epoch_comparison.json').write_text(json.dumps(comparison_record,ensure_ascii=False,indent=2),encoding='utf-8')


### 7.2 응력 분포의 변화


### 05장 / 셀 30 / Notebook


In [ ]:
# 같은 눈금에서 정답과 두 실행의 응력 예측을 비교합니다.
selected_epochs=[summary_100['best_epoch'],summary_1000['best_epoch']]
stress_comparison = plot_comparison(validation_case,comparison_outputs,education_dir,
    run_dir_1000 / 'stress_epoch_comparison.png',selected_epochs,field='stress')
display(Image(filename=stress_comparison['image']))

for label, row in zip(['100에폭 실행에서 검증으로 선택한 모델', '1,000에폭 실행에서 검증으로 선택한 모델'], comparison_rows):
    print(f"{label}: 전체 노드 응력 MAE {row['stress_mae_MPa']:.6f} MPa | 고응력 영역 MAE {row['stress_top5pct_mae_MPa']:.6f} MPa")

# 각 분포에서 최대응력과 그 위치를 찾습니다. 배열 순서 대신 원래 노드 ID를 표시합니다.
import numpy as np
stress_fields = [validation_case.stress[:, 0],
                 *[output['stress'][:, 0] for output in comparison_outputs]]
comparison_labels = ['ANSYS 정답', '100에폭 실행에서 검증으로 선택한 모델', '1,000에폭 실행에서 검증으로 선택한 모델']
target_stress_index = int(np.argmax(stress_fields[0]))
for label, values in zip(comparison_labels, stress_fields):
    index = int(np.argmax(values))
    x, y, z = validation_case.coords[index]
    value_difference = abs(float(values[index]) - float(stress_fields[0][target_stress_index]))
    position_distance = np.linalg.norm(validation_case.coords[index] - validation_case.coords[target_stress_index])
    print(f'{label}: 최대응력 {values[index]:.6f} MPa')
    print(f'  노드 ID {validation_case.node_ids[index]} | 좌표(mm): X={x:.6f}, Y={y:.6f}, Z={z:.6f}')
    print(f'  정답 대비 최대값 절대차: {value_difference:.6f} MPa | 최대 위치 거리: {position_distance:.6f} mm')


### 7.3 변위 분포의 변화


### 05장 / 셀 32 / Notebook


In [ ]:
# 같은 눈금에서 정답과 두 실행의 변위 벡터 크기를 비교합니다.
displacement_comparison = plot_comparison(validation_case,comparison_outputs,education_dir,
    run_dir_1000 / 'displacement_epoch_comparison.png',selected_epochs,field='displacement')
display(Image(filename=displacement_comparison['image']))

for label, row in zip(['100에폭 실행에서 검증으로 선택한 모델', '1,000에폭 실행에서 검증으로 선택한 모델'], comparison_rows):
    print(f"{label}: 전체 노드 변위 크기 MAE {row['displacement_magnitude_mae_mm']:.6f} mm")

# X/Y/Z 변위를 합친 벡터 크기에서 최대 변위 크기와 그 위치를 찾습니다.
import numpy as np
displacement_fields = [np.linalg.norm(validation_case.displacement, axis=1),
                       *[np.linalg.norm(output['displacement'], axis=1) for output in comparison_outputs]]
displacement_labels = ['ANSYS 정답', '100에폭 실행에서 검증으로 선택한 모델', '1,000에폭 실행에서 검증으로 선택한 모델']
target_displacement_index = int(np.argmax(displacement_fields[0]))
for label, values in zip(displacement_labels, displacement_fields):
    index = int(np.argmax(values))
    x, y, z = validation_case.coords[index]
    value_difference = abs(float(values[index]) - float(displacement_fields[0][target_displacement_index]))
    position_distance = np.linalg.norm(validation_case.coords[index] - validation_case.coords[target_displacement_index])
    print(f'{label}: 최대 변위 크기 {values[index]:.6f} mm')
    print(f'  노드 ID {validation_case.node_ids[index]} | 좌표(mm): X={x:.6f}, Y={y:.6f}, Z={z:.6f}')
    print(f'  정답 대비 최대값 절대차: {value_difference:.6f} mm | 최대 위치 거리: {position_distance:.6f} mm')


## 8. 비교 확장: 학습 노드 수에 따른 결과 비교


### 8.1 케이스 하나의 전체 노드를 함께 사용하기


### 05장 / 셀 35 / Notebook


In [ ]:
from course_utils.training_transolver_nodes import train_node_model, node_loaders
from course_utils.transolver_node_compare import (
    check_node_comparison, load_node_models, plot_node_comparison, print_node_extrema)

# 두 실행은 한 번에 케이스 1개씩, 같은 1,000에폭을 학습합니다.
node_config = replace(config, batch_size=1, max_epochs=1000, log_every=100)
node_stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
node_run_sampled = education_dir / 'runs/training' / ('student_nodes_sampled_'+node_stamp)
node_run_full = education_dir / 'runs/training' / ('student_nodes_full_'+node_stamp)
print('학습 케이스별 전체 노드 수:', {cid: len(cases[cid].coords) for cid in split['train']})
preview_datasets, preview_loaders = node_loaders(cases, split, normalizer, fixed, node_config, 'full')
full_batch = next(iter(preview_loaders['train']))
cid = full_batch['case_id'][0]
print('전체 노드 배치 예:', cid, '| 좌표:', tuple(full_batch['coords'].shape))
assert full_batch['coords'].shape == (1, len(cases[cid].coords), 3)
assert set(preview_loaders) == {'train', 'validation'}


### 8.2 케이스 1개씩, 4,096개 노드로 학습


### 05장 / 셀 37 / Notebook


In [ ]:
node_summary_sampled = train_node_model(
    education_dir, node_run_sampled, mode='sampled', config=node_config, device='cuda')
print('4,096개 노드 학습 폴더:', node_run_sampled)
print('검증으로 선택한 에폭:', node_summary_sampled['best_epoch'])


### 8.3 케이스 1개씩, 전체 노드로 학습


### 05장 / 셀 39 / Notebook


In [ ]:
node_summary_full = train_node_model(
    education_dir, node_run_full, mode='full', config=node_config, device='cuda')
print('전체 노드 학습 폴더:', node_run_full)
print('검증으로 선택한 에폭:', node_summary_full['best_epoch'])


### 8.4 같은 기준으로 숫자 비교


### 05장 / 셀 41 / Notebook


In [ ]:
node_checks = check_node_comparison(node_run_sampled, node_run_full)
print('비교 조건 확인:', node_checks)
print('지표 | 4,096개 노드 학습 | 전체 노드 학습')
for label, key in [('검증으로 선택한 에폭', 'best_epoch'),
                   ('고정 검증 총손실', 'best_validation_loss'),
                   ('학습/검증/저장 시간(초)', 'elapsed_seconds'),
                   ('GPU 최대 추가 할당 메모리(MiB)', 'incremental_peak_allocated_MiB')]:
    a, b = node_summary_sampled[key], node_summary_full[key]
    if key == 'best_validation_loss':
        a, b = a['total'], b['total']
    print(f'{label} | {a:.6g} | {b:.6g}')
for key, label in [('stress_MPa', '고정 검증 응력 MAE(MPa)'),
                   *[(f'displacement_{axis}_mm', f'고정 검증 변위 {axis} MAE(mm)') for axis in 'XYZ']]:
    a, b = node_summary_sampled['best_validation_mae'][key], node_summary_full['best_validation_mae'][key]
    print(f'{label} | {a:.6g} | {b:.6g}')

# 두 검증 케이스 모두 전체 노드를 함께 전달합니다. 시험 자료는 평가하지 않습니다.
node_models, node_normalizer = load_node_models(education_dir, node_run_sampled, node_run_full, device)
node_outputs_by_case, node_rows_by_case = {}, {}
for cid in split['validation']:
    outputs, rows = compare_case(*node_models, node_normalizer, cases[cid])
    node_outputs_by_case[cid], node_rows_by_case[cid] = outputs, rows
    print(f"{cid} 전체 {len(cases[cid].coords)}개 노드: 응력 MAE(MPa) {rows[0]['stress_mae_MPa']:.6f} | {rows[1]['stress_mae_MPa']:.6f}")
    print(f"  변위 벡터 크기 MAE(mm) {rows[0]['displacement_magnitude_mae_mm']:.6f} | {rows[1]['displacement_magnitude_mae_mm']:.6f}")
node_case = cases[split['validation'][0]]
node_outputs = node_outputs_by_case[node_case.case_id]
node_selected_epochs = [node_summary_sampled['best_epoch'], node_summary_full['best_epoch']]


### 8.5 응력 분포와 최대값/위치 비교


### 05장 / 셀 43 / Notebook


In [ ]:
node_stress_figure = plot_node_comparison(node_case, node_outputs, education_dir,
    node_run_full / 'nodes_stress_comparison.png', node_selected_epochs, field='stress')
display(Image(filename=node_stress_figure['image']))
node_stress_extrema = print_node_extrema(node_case, node_outputs, 'stress')


### 8.6 변위 분포와 최대값/위치 비교


### 05장 / 셀 45 / Notebook


In [ ]:
node_displacement_figure = plot_node_comparison(node_case, node_outputs, education_dir,
    node_run_full / 'nodes_displacement_comparison.png', node_selected_epochs, field='displacement')
display(Image(filename=node_displacement_figure['image']))
node_displacement_extrema = print_node_extrema(node_case, node_outputs, 'displacement')
node_comparison_record = dict(conditions=node_checks, runs=[str(node_run_sampled), str(node_run_full)],
    validation_case_metrics=node_rows_by_case, stress_extrema=node_stress_extrema,
    displacement_extrema=node_displacement_extrema, test_scored=False)
(node_run_full / 'node_comparison.json').write_text(
    json.dumps(node_comparison_record, ensure_ascii=False, indent=2), encoding='utf-8')


## 9. 결과 정리와 6장 준비


### 9.3 6장에서 평가할 모델 지정


### 05장 / 셀 48 / Notebook


In [ ]:
from course_utils.training_handoff import save_training_handoff
handoff_path = save_training_handoff(education_dir, run_dir_1000)
print('6장에 전달할 학습 폴더:', run_dir_1000.resolve())
print('학습 연결 정보:', handoff_path)
print('추론 방식: 케이스 전체 노드를 함께 전달')


### 9.4 Console 실행과 문제 해결


### 05장 / 셀 49 / Console 01


```bat
python education/preparation/train_transolver.py --output-dir education/runs/training/student_transolver_console_01 --max-epochs 1000 --device cuda
```
